# 005-village-crossdate

# Imports

In [ ]:
import json
import os

os.environ.setdefault("PYTORCH_ENABLE_MPS_FALLBACK", "1")

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from uavloc.data.drone_frames import DroneFrame, load_frames
from uavloc.data.satellite import GeoImage
from uavloc.datasets import dataset_ref
from uavloc.features.factory import build_features, correlator_config
from uavloc.geo import LatLon
from uavloc.matching.benchmark import run_benchmark, summarise
from uavloc.matching.correlator import PositionCorrelator
from uavloc.track import REPO_ROOT, experiment_dir, log_run, set_seed

# Params

In [ ]:
EXP = "005-village-crossdate"
METHOD = "segmentation"           # "vgg11" | "segmentation"
MATCHING = "weighted-ncc"         # "cosine" (2025) | "weighted-ncc"
CLASS_WEIGHTS = (0.0, 1.0, 1.0, 0.5, 0.0, 0.5)   # background, building, track, woody, field, water; weighted-ncc only
NOTE = ""

NOISE_M = [5.0, 10.0, 15.0]
PRIMARY_NOISE_M = 15.0
N_DIRECTIONS = 8
SEED = 42
DEVICE = "mps"

SEARCH_SCALE = 1.7
FOV_W_DEG = 60.0
SAT_FEATURE_CROP_PCT = 0.03
DRONE_FEATURE_CROP_PCT = 0.0

VGG11_CHECKPOINT = "../archive/uav_localization_2025/feature_models/models_finetrain/vgg11_drop_margin08_adamw_model_20250521_034900.pth"
DRONE_INPUT_PX = 448

SEG_ARCHITECTURE = "lraspp"
SEG_CHECKPOINT_SAT = "pocs/mobilenet-lraspp-speed/model-lraspp-bn-satellite.pt"
SEG_CHECKPOINT_DRONE = "pocs/mobilenet-lraspp-speed/model-lraspp-bn-satellite.pt"   # the "drone" view is satellite imagery
SEG_CHANNELS = "all"
DRONE_SEG_GSD_M = 0.385           # native tile resolution: no resampling of the "drone" view
MATCH_GSD_M = 0.385

FRAMES_DATASET = "village-crossdate-frames"

# Data

In [ ]:
set_seed(SEED)

In [ ]:
RUN_CONFIG = {
    "method": METHOD, "matching": MATCHING, "class_weights": CLASS_WEIGHTS,
    "noise_m": NOISE_M, "primary_noise_m": PRIMARY_NOISE_M, "n_directions": N_DIRECTIONS,
    "search_scale": SEARCH_SCALE, "fov_w_deg": FOV_W_DEG,
    "sat_feature_crop_pct": SAT_FEATURE_CROP_PCT, "drone_feature_crop_pct": DRONE_FEATURE_CROP_PCT,
    "vgg11_checkpoint": VGG11_CHECKPOINT, "drone_input_px": DRONE_INPUT_PX,
    "seg_architecture": SEG_ARCHITECTURE, "seg_checkpoint_sat": SEG_CHECKPOINT_SAT, "seg_checkpoint_drone": SEG_CHECKPOINT_DRONE,
    "seg_channels": SEG_CHANNELS, "drone_seg_gsd_m": DRONE_SEG_GSD_M, "match_gsd_m": MATCH_GSD_M,
    "frames_dataset": FRAMES_DATASET, "frames_dataset_ref": dataset_ref(FRAMES_DATASET),
}

In [ ]:
def frame_meta(frame: DroneFrame) -> dict:
    return json.loads((frame.image_path.parent / "meta.json").read_text())

In [ ]:
frames = load_frames(FRAMES_DATASET)
places = {}
for frame in frames:
    places.setdefault(frame_meta(frame)["place"], []).append(frame)
{place: len(members) for place, members in places.items()}

In [ ]:
def place_map(place_frames: list[DroneFrame]) -> GeoImage:
    """The stale date of the place's village, the map every frame there is matched against."""
    meta = frame_meta(place_frames[0])
    return GeoImage.load(REPO_ROOT / meta["map_image"], LatLon(*meta["map_top_left"]), LatLon(*meta["map_bottom_right"]))

# Run

In [ ]:
def run_place(place_frames: list[DroneFrame]) -> pd.DataFrame:
    """Build features on this place's map and run the injected-error benchmark on its frames."""
    features = build_features(RUN_CONFIG, place_map(place_frames), DEVICE)
    correlator = PositionCorrelator(features, correlator_config(RUN_CONFIG))
    return pd.concat([run_benchmark(correlator, place_frames, noise_m, N_DIRECTIONS, SEED) for noise_m in NOISE_M],
                     ignore_index=True)

In [ ]:
records = pd.concat([run_place(members).assign(place=place) for place, members in places.items()], ignore_index=True)
records.groupby(["place", "noise_m"])[["error_m", "reachable"]].mean()

In [ ]:
subsets = {"all": {f.name for f in frames}, **{place.split("-")[0]: {f.name for f in members} for place, members in places.items()}}
metrics = summarise(records, subsets, PRIMARY_NOISE_M)
{k: v for k, v in metrics.items() if "_at_" not in k}

# Plots

In [ ]:
fig_cdf, ax = plt.subplots(figsize=(6, 4))
for noise_m, group in records.groupby("noise_m"):
    errors = np.sort(group["error_m"].to_numpy())
    line, = ax.plot(errors, np.linspace(0, 1, len(errors)), label=f"{noise_m:g} m injected")
    ax.axvline(noise_m, color=line.get_color(), ls=":", lw=1)
ax.set_xlabel("position error after correction, m"); ax.set_ylabel("share of draws")
ax.set_title(f"{METHOD}, {MATCHING}"); ax.legend(); ax.grid(alpha=.3)
fig_cdf.tight_layout()

In [ ]:
primary = records[records["noise_m"] == PRIMARY_NOISE_M]
subset_means = {name: primary[primary["frame"].isin(members)]["error_m"].mean() for name, members in subsets.items()}
fig_subsets, ax = plt.subplots(figsize=(6, 3))
ax.bar(list(subset_means), list(subset_means.values()))
ax.axhline(PRIMARY_NOISE_M, color="k", ls=":", lw=1, label="no correction")
ax.set_ylabel(f"mean error @ {PRIMARY_NOISE_M:g} m, m"); ax.set_title(f"{METHOD}, {MATCHING}"); ax.legend()
fig_subsets.tight_layout()

In [ ]:
fig_reach, ax = plt.subplots(figsize=(6, 3.5))
for reachable, group in primary.groupby("reachable"):
    ax.hist(group["error_m"], bins=30, range=(0, 40), alpha=.6, label=f"truth in window: {reachable}")
ax.axvline(PRIMARY_NOISE_M, color="k", ls=":", lw=1)
ax.set_xlabel(f"error @ {PRIMARY_NOISE_M:g} m injected, m"); ax.set_ylabel("draws"); ax.set_title(f"{METHOD}, {MATCHING}"); ax.legend()
fig_reach.tight_layout()

In [ ]:
run = log_run(
    exp=EXP,
    config=RUN_CONFIG,
    metrics=metrics,
    note=NOTE,
    figures=[fig_cdf, fig_subsets, fig_reach],
    seed=SEED,
    device=DEVICE,
)

In [ ]:
artifacts = experiment_dir(EXP) / "artifacts"
artifacts.mkdir(exist_ok=True)
records.to_csv(artifacts / f"records-{run['run']}.csv", index=False)